In [1]:
import pymongo

def fix_flota_data_types():
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    db = client["Espana"]
    collection = db["airline_stats"]
    
    # 1. Finde alle Dokumente, bei denen 'flota' ein String (BSON Type 2) ist
    string_docs = list(collection.find({"flota": {"$type": 2}}))
    
    if not string_docs:
        print("✅ Keine ungültigen Datentypen gefunden. Alles bereits im Array-Format.")
        return

    print(f"🔍 Gefunden: {len(string_docs)} Dokumente mit String-Format in 'flota'.")
    
    for doc in string_docs:
        old_value = doc["flota"]
        # Wir wandeln den String in ein Array mit einem Element um
        new_array = [old_value] if old_value else []
        
        collection.update_one(
            {"_id": doc["_id"]},
            {"$set": {"flota": new_array}}
        )
        print(f"🔄 {doc.get('airline', 'Unbekannt')}: String '{old_value}' -> Array {new_array}")

    print("🚀 Bereinigung abgeschlossen.")
fix_flota_data_types()

✅ Keine ungültigen Datentypen gefunden. Alles bereits im Array-Format.


In [2]:
import pymongo

def get_fleet_statistics():
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    db = client["Espana"]
    
    pipeline = [
        # 1. Nur Dokumente nehmen, die ein Flotten-Array haben
        {"$match": {"flota": {"$type": "array"}}},
        
        # 2. Das Array 'flota' in einzelne Dokumente zerlegen
        {"$unwind": "$flota"},
        
        # 3. Gruppieren nach Airline und die Anzahl zählen
        {"$group": {
            "_id": "$airline",
            "anzahl_flugzeuge": {"$sum": 1},
            "modelle": {"$addToSet": "$flota"}
        }},
        
        # 4. Sortieren nach Anzahl (absteigend)
        {"$sort": {"anzahl_flugzeuge": -1}}
    ]
    
    results = list(db.airline_stats.aggregate(pipeline))
    
    print("--- ✈️ Globale Flotten-Übersicht ---")
    for res in results:
        print(f"Airline: {res['_id']}")
        print(f"Gesamtanzahl: {res['anzahl_flugzeuge']} Maschinen")
        print(f"Modell-Mix: {', '.join(res['modelle'])}")
        print("-" * 30)

get_fleet_statistics()

--- ✈️ Globale Flotten-Übersicht ---
Airline: Vueling
Gesamtanzahl: 1 Maschinen


TypeError: sequence item 0: expected str instance, list found